# 📊 Retail Sales & Demand Forecasting
## FC Raffa Rafli — Retail Analytics Portfolio

---

1. **Exploratory Data Analysis (EDA)** — memahami pola data sebelum membuat model
2. **Uji Statistik** — ADF test untuk stasioneritas, ACF/PACF untuk autokorelasi
3. **4 Baseline Models** — Mean, Naive, Seasonal Naive, Moving Average
4. **2 Statistical Models** — Holt-Winters (Exponential Smoothing) & SARIMA
5. **Model Evaluation** — MAE, RMSE, WAPE (bukan MAPE — akan dijelaskan kenapa)
6. **Model Selection** — memilih model terbaik berdasarkan data, bukan kompleksitas
7. **Final Forecast** — prediksi 28 hari ke depan (Februari 2026) + confidence interval
8. **Business Interpretation** — menerjemahkan angka menjadi keputusan bisnis

### Data Source
- **Input:** `data/processed/vw_daily_kpi.csv` (export dari PostgreSQL view `analytics.vw_daily_kpi`)
- **Period:** 2025-09-01 → 2026-01-31 (153 hari, termasuk 8 hari tanpa penjualan)
- **Target 1:** `order_count` — jumlah order harian
- **Target 2:** `revenue` — pendapatan harian (Rp)

### Output
- `data/processed/forecast_daily.csv` — 28 baris prediksi Feb 2026
- `data/processed/forecast_model_evaluation.csv` — tabel perbandingan 6 model × 2 target

## 1️⃣ Setup — Import Library & Konfigurasi

Library yang digunakan:
- **pandas** — manipulasi data tabular
- **numpy** — operasi numerik / array
- **matplotlib + seaborn** — visualisasi data
- **statsmodels** — model statistik (Holt-Winters, SARIMA, ADF test, ACF/PACF, seasonal decomposition)
- **warnings** — menyembunyikan warning yang tidak relevan

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Konfigurasi tampilan ──
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Path (robust whether run from project root or python/ directory) ──
if os.path.exists(os.path.join('data', 'processed', 'vw_daily_kpi.csv')):
    DATA_DIR = os.path.join('data', 'processed')
else:
    DATA_DIR = os.path.join('..', 'data', 'processed')
INPUT_FILE = os.path.join(DATA_DIR, 'vw_daily_kpi.csv')

# ── Konstanta ──
HOLDOUT_DAYS    = 28     # Validasi: 28 hari terakhir
FORECAST_DAYS   = 28     # Prediksi: 28 hari Feb 2026
SEASONAL_PERIOD = 7      # Pola musiman mingguan
TARGETS = ['order_count', 'revenue']

print("✅ Library loaded!")

## 2️⃣ Load Data & Inspeksi Awal

Langkah pertama dalam setiap proyek data science: **kenali datamu**.

Kita akan:
1. Load CSV dan parse kolom tanggal
2. Pastikan data terurut berdasarkan tanggal
3. Buat date range lengkap (setiap hari kalender harus ada, meskipun zero sales)
4. Lihat `.info()` dan `.describe()` untuk memahami distribusi

In [ ]:
# Load data
df = pd.read_csv(INPUT_FILE, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Baris awal: {len(df)}")
print(f"Tanggal:    {df['date'].min().date()} → {df['date'].max().date()}")
print()

# Reindex: pastikan SETIAP tanggal ada (tidak ada gap)
full_range = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
df = (df.set_index('date')
        .reindex(full_range)
        .rename_axis('date')
        .reset_index())

# Isi NaN dengan 0 untuk hari tanpa penjualan
df['order_count'] = df['order_count'].fillna(0).astype(int)
df['revenue']     = df['revenue'].fillna(0.0).astype(float)

print(f"Baris setelah reindex: {len(df)}")
print(f"Zero-sales days: {(df['order_count'] == 0).sum()}")
print()

# Inspeksi
df.info()
print()
df.describe()

## 3️⃣ Data Completeness Check

Sebelum modeling, kita harus memastikan:
- **Tidak ada gap tanggal** — setiap hari kalender harus ada
- **Zero-sales days terisi 0** — bukan NULL atau dihapus (ini penting untuk time-series!)
- **Tidak ada duplikat tanggal**

> ⚠️ **Kenapa ini penting?**
> Model time-series (SARIMA, Holt-Winters) membutuhkan data dengan frekuensi tetap (harian).
> Jika ada tanggal yang hilang, model akan salah menghitung pola musiman.

In [ ]:
# Cek gap tanggal
expected_days = (df['date'].max() - df['date'].min()).days + 1
actual_days   = len(df)
print(f"Expected days: {expected_days}")
print(f"Actual days:   {actual_days}")
print(f"Gap:           {expected_days - actual_days} hari")

# Cek duplikat
duplicates = df['date'].duplicated().sum()
print(f"Duplikat:      {duplicates}")

# Cek zero-sales days
zero_days = df[df['order_count'] == 0]
print(f"\nZero-sales days ({len(zero_days)} hari):")
print(zero_days[['date', 'order_count', 'revenue']].to_string(index=False))

# Cek bulan pertama & terakhir
print(f"\nBulan pertama: {df['date'].min().strftime('%B %Y')}")
print(f"Bulan terakhir: {df['date'].max().strftime('%B %Y')}")
print(f"Total bulan: ~{((df['date'].max() - df['date'].min()).days) / 30:.1f}")
print("\n✅ Data completeness check passed!")

---
# 📈 BAGIAN 2 — Exploratory Data Analysis (EDA)

EDA adalah langkah **paling penting** sebelum membangun model forecasting.
Tujuannya adalah memahami:
1. **Trend** — apakah data naik, turun, atau stabil?
2. **Seasonality** — apakah ada pola mingguan?
3. **Outlier** — apakah ada lonjakan/anomali?
4. **Distribusi** — bagaimana sebaran data?

> 💡 **Tips Data Scientist:** Jangan pernah langsung lompat ke modeling tanpa EDA.
> Seringkali insight terbaik ditemukan di tahap ini, bukan di model.

### 📉 Time Series Plot — Order Count & Revenue

Plot pertama yang wajib dibuat: **melihat data secara kronologis**.
Ini memberi gambaran awal tentang trend dan pola musiman.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Order Count
axes[0].plot(df['date'], df['order_count'], color='#2196F3', linewidth=1.2, alpha=0.9)
axes[0].fill_between(df['date'], df['order_count'], alpha=0.15, color='#2196F3')
axes[0].set_ylabel('Order Count', fontsize=12)
axes[0].set_title('Daily Order Count — Sep 2025 to Jan 2026', fontsize=14, fontweight='bold')
axes[0].axhline(df['order_count'].mean(), color='red', linestyle='--', alpha=0.5, label=f"Mean: {df['order_count'].mean():.1f}")
axes[0].legend()

# Revenue
axes[1].plot(df['date'], df['revenue'], color='#4CAF50', linewidth=1.2, alpha=0.9)
axes[1].fill_between(df['date'], df['revenue'], alpha=0.15, color='#4CAF50')
axes[1].set_ylabel('Revenue (Rp)', fontsize=12)
axes[1].set_title('Daily Revenue — Sep 2025 to Jan 2026', fontsize=14, fontweight='bold')
axes[1].axhline(df['revenue'].mean(), color='red', linestyle='--', alpha=0.5, label=f"Mean: Rp{df['revenue'].mean():,.0f}")
axes[1].legend()

axes[1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 📊 Pola Mingguan (Day-of-Week Seasonality)

Apakah ada hari tertentu yang konsisten lebih ramai atau sepi?
Boxplot membantu melihat **distribusi per hari**, bukan hanya rata-rata.

> 💡 Jika ada pola mingguan yang jelas, model dengan `seasonal_period=7` akan cocok.

In [ ]:
df['day_name'] = df['date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Order Count by Day of Week
sns.boxplot(data=df, x='day_name', y='order_count', order=day_order,
            palette='Blues_d', ax=axes[0])
axes[0].set_title('Order Count by Day of Week', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Order Count')

# Revenue by Day of Week
sns.boxplot(data=df, x='day_name', y='revenue', order=day_order,
            palette='Greens_d', ax=axes[1])
axes[1].set_title('Revenue by Day of Week', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue (Rp)')

plt.tight_layout()
plt.show()

# Statistik per hari
print("Rata-rata Order per Hari:")
print(df.groupby('day_name')['order_count'].mean().reindex(day_order).round(1).to_string())

### 📊 Aggregasi Bulanan

Melihat data pada level bulan membantu mengidentifikasi **trend jangka menengah**.

In [ ]:
monthly = df.set_index('date').resample('MS').agg({
    'order_count': 'sum',
    'revenue': 'sum'
}).reset_index()
monthly['month_label'] = monthly['date'].dt.strftime('%b %Y')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly Orders
bars1 = axes[0].bar(monthly['month_label'], monthly['order_count'], color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_title('Monthly Total Orders', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Orders')
for bar, val in zip(bars1, monthly['order_count']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{int(val)}', ha='center', va='bottom', fontsize=10)

# Monthly Revenue
bars2 = axes[1].bar(monthly['month_label'], monthly['revenue']/1000, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('Monthly Revenue (Ribu Rp)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Revenue (Rp 000)')
for bar, val in zip(bars2, monthly['revenue']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'Rp{val/1000:,.0f}K', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### 📊 Distribusi Data (Histogram + KDE)

Melihat apakah data berdistribusi normal atau skewed (condong).
Ini membantu memahami apakah mean adalah representasi yang baik.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Order Count Distribution
axes[0].hist(df['order_count'], bins=20, color='#2196F3', alpha=0.7, edgecolor='white', density=True)
df['order_count'].plot.kde(ax=axes[0], color='darkblue', linewidth=2)
axes[0].axvline(df['order_count'].mean(), color='red', linestyle='--', label=f"Mean: {df['order_count'].mean():.1f}")
axes[0].axvline(df['order_count'].median(), color='orange', linestyle='--', label=f"Median: {df['order_count'].median():.1f}")
axes[0].set_title('Distribution: Order Count', fontsize=13, fontweight='bold')
axes[0].legend()

# Revenue Distribution
axes[1].hist(df['revenue'], bins=20, color='#4CAF50', alpha=0.7, edgecolor='white', density=True)
df['revenue'].plot.kde(ax=axes[1], color='darkgreen', linewidth=2)
axes[1].axvline(df['revenue'].mean(), color='red', linestyle='--', label=f"Mean: Rp{df['revenue'].mean():,.0f}")
axes[1].axvline(df['revenue'].median(), color='orange', linestyle='--', label=f"Median: Rp{df['revenue'].median():,.0f}")
axes[1].set_title('Distribution: Revenue', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

### 📊 Autocorrelation (ACF) & Partial Autocorrelation (PACF)

Ini adalah analisis **KUNCI** untuk memilih parameter ARIMA.

- **ACF (Autocorrelation Function):** Menunjukkan korelasi antara data hari ini dengan data N hari lalu
  - Jika ada spike di lag 7, 14, 21 → ada **pola mingguan**
  - ACF membantu menentukan parameter **q** (MA) dan **Q** (seasonal MA)

- **PACF (Partial Autocorrelation Function):** Menunjukkan korelasi "murni" setelah menghilangkan efek lag di antaranya
  - PACF membantu menentukan parameter **p** (AR) dan **P** (seasonal AR)

> 💡 **Cara baca:** Garis biru yang melewati area abu-abu (confidence band) menandakan lag yang signifikan secara statistik.

In [ ]:
for target in TARGETS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    plot_acf(df[target].values, lags=40, ax=axes[0], title=f'ACF — {target}',
             color='#2196F3', vlines_kwargs={'color': '#2196F3'})
    plot_pacf(df[target].values, lags=40, ax=axes[1], title=f'PACF — {target}',
              color='#E91E63', vlines_kwargs={'color': '#E91E63'}, method='ywm')

    # Highlight lag 7, 14, 21
    for ax in axes:
        for lag in [7, 14, 21]:
            ax.axvline(lag, color='gray', alpha=0.3, linestyle=':')

    plt.suptitle(f'Autocorrelation Analysis: {target}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

### 🧪 Augmented Dickey-Fuller Test (Uji Stasioneritas)

SARIMA membutuhkan data yang **stasioner** (mean & variance konstan sepanjang waktu).

- **H0 (Null Hypothesis):** Data TIDAK stasioner (ada unit root)
- **H1 (Alternative):** Data stasioner

Keputusan:
- Jika **p-value < 0.05** → tolak H0 → data **stasioner** ✅
- Jika **p-value ≥ 0.05** → gagal tolak H0 → data **tidak stasioner** → perlu differencing (d=1)

> 💡 Jika data sudah stasioner, parameter `d=0`. Jika tidak, gunakan `d=1` (atau `D=1` untuk seasonal).

In [ ]:
for target in TARGETS:
    result = adfuller(df[target].values, autolag='AIC')
    print(f"{'═'*50}")
    print(f"ADF Test: {target}")
    print(f"{'═'*50}")
    print(f"  Test Statistic: {result[0]:.4f}")
    print(f"  p-value:        {result[1]:.6f}")
    print(f"  Lags Used:      {result[2]}")
    print(f"  Observations:   {result[3]}")
    print(f"  Critical Values:")
    for key, val in result[4].items():
        print(f"    {key}: {val:.4f}")

    if result[1] < 0.05:
        print(f"  ✅ STASIONER (p={result[1]:.4f} < 0.05) → d=0 bisa dipertimbangkan")
    else:
        print(f"  ❌ TIDAK STASIONER (p={result[1]:.4f} ≥ 0.05) → perlu differencing (d=1)")
    print()

### 🔬 Seasonal Decomposition (Dekomposisi Musiman)

Dekomposisi memecah time-series menjadi 3 komponen:
1. **Trend** — arah pergerakan jangka panjang
2. **Seasonal** — pola berulang (mingguan dalam kasus kita)
3. **Residual** — sisa yang tidak bisa dijelaskan oleh trend & seasonal

> 💡 Jika komponen **Seasonal** menunjukkan pola yang jelas dan berulang, maka model
> dengan seasonal component (Holt-Winters, SARIMA) akan lebih baik dari baseline sederhana.

In [ ]:
for target in TARGETS:
    series = df.set_index('date')[target].asfreq('D').fillna(0)
    decomposition = seasonal_decompose(series, model='additive', period=7)

    fig, axes = plt.subplots(4, 1, figsize=(16, 10))

    decomposition.observed.plot(ax=axes[0], color='#333', title='Observed')
    decomposition.trend.plot(ax=axes[1], color='#E91E63', title='Trend')
    decomposition.seasonal.plot(ax=axes[2], color='#2196F3', title='Seasonal (Weekly Pattern)')
    decomposition.resid.plot(ax=axes[3], color='#FF9800', title='Residual')

    fig.suptitle(f'Seasonal Decomposition: {target} (period=7)', fontsize=15, fontweight='bold', y=1.01)
    for ax in axes:
        ax.set_xlabel('')
    plt.tight_layout()
    plt.show()
    print()

---
# 🔪 BAGIAN 3 — Train/Test Split & Baseline Models

## ⚠️ ATURAN PENTING: Time-Series TIDAK BOLEH Di-split Random!

Berbeda dengan klasifikasi/regresi biasa, time-series memiliki **urutan kronologis** yang harus dijaga.

**SALAH:** `train_test_split(X, y, test_size=0.2, random_state=42)` — ini mengacak urutan!

**BENAR:** Holdout split berdasarkan waktu:
- **Train** = data awal (Sep 2025 → awal Jan 2026)
- **Test/Validation** = 28 hari terakhir (Jan 2026)

Kenapa 28 hari? Karena forecast horizon kita juga 28 hari (Feb 2026),
sehingga evaluasi pada holdout set benar-benar mensimulasikan situasi riil.

In [ ]:
# ── Holdout Split ──
cutoff = len(df) - HOLDOUT_DAYS
train = df.iloc[:cutoff].copy()
test  = df.iloc[cutoff:].copy()

print(f"Train: {len(train)} hari ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Test:  {len(test)} hari ({test['date'].min().date()} → {test['date'].max().date()})")
print(f"Ratio: {len(train)/len(df)*100:.1f}% / {len(test)/len(df)*100:.1f}%")

# Visualisasi split
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(train['date'], train['order_count'], color='#2196F3', label='Train', linewidth=1.5)
ax.plot(test['date'], test['order_count'], color='#E91E63', label='Test (Holdout)', linewidth=1.5)
ax.axvline(test['date'].min(), color='red', linestyle='--', alpha=0.7, label='Split Point')
ax.fill_between(test['date'], test['order_count'], alpha=0.2, color='#E91E63')
ax.set_title('Train / Test Split — Order Count', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 📏 Metrics: Kenapa WAPE dan Bukan MAPE?

Kita menggunakan 3 metrik evaluasi:

| Metrik | Rumus | Kelebihan | Kekurangan |
|--------|-------|-----------|------------|
| **MAE** | `mean(|actual - predicted|)` | Mudah dipahami (dalam satuan asli) | Tidak relatif |
| **RMSE** | `sqrt(mean((actual - predicted)²))` | Menghukum error besar lebih keras | Sensitif outlier |
| **WAPE** | `sum(|actual - predicted|) / sum(|actual|)` | Relatif, aman untuk zero values | Bisa dominasi oleh hari besar |

> ⚠️ **Kenapa BUKAN MAPE?**
> MAPE = `mean(|actual - predicted| / actual)` akan menjadi **infinity** jika `actual = 0`.
> Karena dataset kita punya 8 zero-sales days, MAPE tidak valid. WAPE adalah alternatif yang lebih robust.

In [ ]:
# ── Fungsi Metrik ──
def calc_mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

def calc_rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

def calc_wape(actual, predicted):
    denom = np.sum(np.abs(actual))
    return np.sum(np.abs(actual - predicted)) / denom if denom > 0 else np.inf

print("✅ Metric functions defined!")

## 🔹 Baseline 1: Mean Baseline

**Konsep:** Prediksi = rata-rata historis (training set) untuk setiap hari.

Ini adalah model paling sederhana. Jika model canggih tidak bisa mengalahkan ini, berarti data kita terlalu noisy untuk diprediksi.

> 💡 **Analogi:** Seperti menebak ujian besok dengan nilai rata-rata ujian sebelumnya.

In [ ]:
def predict_mean(train_vals, n):
    return np.full(n, train_vals.mean())

# ── Evaluate ──
all_results = []

for target in TARGETS:
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)
    pred = predict_mean(tr, len(te))

    mae  = calc_mae(te, pred)
    rmse = calc_rmse(te, pred)
    wape = calc_wape(te, pred)

    all_results.append({'target': target, 'model': 'Mean Baseline',
                        'mae': mae, 'rmse': rmse, 'wape': wape})
    print(f"{target:15s}  MAE={mae:>10,.2f}  RMSE={rmse:>10,.2f}  WAPE={wape*100:>6.2f}%")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for i, target in enumerate(TARGETS):
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)
    pred = predict_mean(tr, len(te))
    axes[i].plot(test['date'], te, label='Actual', color='#333', linewidth=1.5)
    axes[i].plot(test['date'], pred, label='Mean Baseline', color='red', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'Mean Baseline — {target}', fontweight='bold')
    axes[i].legend()
plt.tight_layout()
plt.show()

## 🔹 Baseline 2: Naive Last-Value

**Konsep:** Prediksi = nilai terakhir dari training set, diulang untuk setiap hari.

> 💡 **Analogi:** "Besok akan sama seperti hari ini." Sangat naif, tapi sering mengejutkan sebagai benchmark.

In [ ]:
def predict_naive(train_vals, n):
    return np.full(n, train_vals[-1])

for target in TARGETS:
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)
    pred = predict_naive(tr, len(te))

    mae  = calc_mae(te, pred)
    rmse = calc_rmse(te, pred)
    wape = calc_wape(te, pred)

    all_results.append({'target': target, 'model': 'Naive Last-Value',
                        'mae': mae, 'rmse': rmse, 'wape': wape})
    print(f"{target:15s}  MAE={mae:>10,.2f}  RMSE={rmse:>10,.2f}  WAPE={wape*100:>6.2f}%")

## 🔹 Baseline 3: Seasonal Naive 7-Day

**Konsep:** Prediksi = nilai dari **hari yang sama minggu lalu**, diulang secara siklik.

Contoh: Prediksi Senin depan = nilai Senin lalu, Selasa depan = Selasa lalu, dst.

> 💡 Ini adalah baseline terbaik jika data memiliki **pola mingguan yang kuat**.

In [ ]:
def predict_seasonal_naive(train_vals, n, period=SEASONAL_PERIOD):
    last_season = train_vals[-period:]
    reps = int(np.ceil(n / period))
    return np.tile(last_season, reps)[:n]

for target in TARGETS:
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)
    pred = predict_seasonal_naive(tr, len(te))

    mae  = calc_mae(te, pred)
    rmse = calc_rmse(te, pred)
    wape = calc_wape(te, pred)

    all_results.append({'target': target, 'model': 'Seasonal Naive 7D',
                        'mae': mae, 'rmse': rmse, 'wape': wape})
    print(f"{target:15s}  MAE={mae:>10,.2f}  RMSE={rmse:>10,.2f}  WAPE={wape*100:>6.2f}%")

## 🔹 Baseline 4: 7-Day Moving Average

**Konsep:** Prediksi = rata-rata 7 hari terakhir dari training set.

Lebih responsif dari Mean Baseline karena hanya mempertimbangkan data terbaru.

> 💡 Model ini baik jika tidak ada trend kuat dan data relatif stabil.

In [ ]:
def predict_ma7(train_vals, n):
    return np.full(n, train_vals[-7:].mean())

for target in TARGETS:
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)
    pred = predict_ma7(tr, len(te))

    mae  = calc_mae(te, pred)
    rmse = calc_rmse(te, pred)
    wape = calc_wape(te, pred)

    all_results.append({'target': target, 'model': '7-Day Moving Average',
                        'mae': mae, 'rmse': rmse, 'wape': wape})
    print(f"{target:15s}  MAE={mae:>10,.2f}  RMSE={rmse:>10,.2f}  WAPE={wape*100:>6.2f}%")

## 📊 Perbandingan 4 Baseline

Sebelum melanjutkan ke model statistik yang lebih kompleks,
mari lihat perbandingan 4 baseline yang sudah kita buat.

In [ ]:
baseline_df = pd.DataFrame(all_results)
baseline_df['wape_pct'] = baseline_df['wape'].apply(lambda x: f"{x*100:.2f}%")

print("\n📊 Baseline Comparison:")
print("="*80)
for target in TARGETS:
    subset = baseline_df[baseline_df['target'] == target].copy()
    subset = subset.sort_values('wape')
    print(f"\n  Target: {target}")
    print(f"  {'Model':<25s} {'MAE':>12s} {'RMSE':>12s} {'WAPE':>10s}")
    print(f"  {'─'*60}")
    for _, row in subset.iterrows():
        marker = ' 🏆' if row['wape'] == subset['wape'].min() else ''
        print(f"  {row['model']:<25s} {row['mae']:>12,.2f} {row['rmse']:>12,.2f} {row['wape']*100:>8.2f}%{marker}")

# Bar chart WAPE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, target in enumerate(TARGETS):
    subset = baseline_df[baseline_df['target'] == target].sort_values('wape')
    colors = ['#4CAF50' if w == subset['wape'].min() else '#90CAF9' for w in subset['wape']]
    axes[i].barh(subset['model'], subset['wape'] * 100, color=colors, edgecolor='white')
    axes[i].set_xlabel('WAPE (%)')
    axes[i].set_title(f'Baseline WAPE: {target}', fontweight='bold')
    for j, (_, row) in enumerate(subset.iterrows()):
        axes[i].text(row['wape'] * 100 + 0.5, j, f"{row['wape']*100:.1f}%", va='center', fontsize=10)
plt.tight_layout()
plt.show()

---
# 🧠 BAGIAN 4 — Statistical Models

Sekarang kita masuk ke model yang lebih canggih:
1. **Holt-Winters (Exponential Smoothing)** — model klasik untuk trend + seasonality
2. **SARIMA** — model time-series paling populer di industri

> ⚠️ **Prinsip Master Plan:** Jika baseline mengalahkan model kompleks → pilih baseline.
> Business value > model complexity.

## 🔸 Model 5: Holt-Winters (Exponential Smoothing)

### Apa itu Holt-Winters?
Holt-Winters adalah pengembangan dari **exponential smoothing** yang menangkap 3 komponen:

1. **Level (α):** Rata-rata terbobot yang memberi bobot lebih pada data terbaru
2. **Trend (β):** Arah pergerakan naik/turun
3. **Seasonal (γ):** Pola berulang dengan period tertentu

### Mode yang Kita Gunakan
- **Trend:** Additive (trend linear)
- **Seasonal:** Additive (pola musiman konstan)
- **Period:** 7 (musiman mingguan)

> 💡 "Additive" artinya komponen dijumlahkan: `Y = Level + Trend + Seasonal + Error`.
> Jika amplitude seasonality berubah seiring trend, gunakan "Multiplicative".

In [ ]:
def fit_holt_winters(train_vals, n, period=SEASONAL_PERIOD):
    idx = pd.date_range('2000-01-01', periods=len(train_vals), freq='D')
    series = pd.Series(train_vals, index=idx)
    try:
        model = ExponentialSmoothing(
            series,
            trend='add',
            seasonal='add',
            seasonal_periods=period,
            initialization_method='estimated'
        ).fit(optimized=True)
        pred = model.forecast(n).values
        return np.maximum(pred, 0), model
    except Exception as e:
        print(f"⚠️ Holt-Winters error: {e}")
        return None, None

# Evaluate
for target in TARGETS:
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)

    pred, model = fit_holt_winters(tr, len(te))
    if pred is not None:
        pred = np.maximum(pred, 0)
        mae  = calc_mae(te, pred)
        rmse = calc_rmse(te, pred)
        wape = calc_wape(te, pred)

        all_results.append({'target': target, 'model': 'Holt-Winters',
                            'mae': mae, 'rmse': rmse, 'wape': wape})
        print(f"{target:15s}  MAE={mae:>10,.2f}  RMSE={rmse:>10,.2f}  WAPE={wape*100:>6.2f}%")

        # Tampilkan parameter optimal
        print(f"  Parameters: α(level)={model.params['smoothing_level']:.4f}, "
              f"β(trend)={model.params['smoothing_trend']:.4f}, "
              f"γ(seasonal)={model.params['smoothing_seasonal']:.4f}")
    print()

# Visualisasi Holt-Winters fit vs actual
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for i, target in enumerate(TARGETS):
    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)
    pred, _ = fit_holt_winters(tr, len(te))
    if pred is not None:
        axes[i].plot(test['date'], te, label='Actual', color='#333', linewidth=1.5)
        axes[i].plot(test['date'], np.maximum(pred, 0), label='Holt-Winters', color='#E91E63', linewidth=1.5)
        axes[i].set_title(f'Holt-Winters — {target}', fontweight='bold')
        axes[i].legend()
plt.tight_layout()
plt.show()

## 🔸 Model 6: SARIMA (Seasonal ARIMA)

### Apa itu SARIMA?
SARIMA = **S**easonal **A**uto**R**egressive **I**ntegrated **M**oving **A**verage

Notasi: **SARIMA(p,d,q)(P,D,Q,m)**

| Parameter | Komponen | Makna |
|-----------|----------|-------|
| **p** | AR (AutoRegressive) | Berapa banyak lag masa lalu yang mempengaruhi hari ini |
| **d** | I (Integrated/Differencing) | Berapa kali differencing agar data stasioner |
| **q** | MA (Moving Average) | Berapa banyak error masa lalu yang mempengaruhi hari ini |
| **P** | Seasonal AR | Sama seperti p, tapi pada level musiman |
| **D** | Seasonal I | Differencing musiman |
| **Q** | Seasonal MA | Sama seperti q, tapi pada level musiman |
| **m** | Period | Panjang siklus musiman (7 = mingguan) |

### Cara Memilih Parameter
1. **d/D:** Dari ADF test — jika data sudah stasioner, d=0; jika belum, d=1
2. **p/P:** Dari PACF plot — lihat di lag mana korelasi signifikan lalu terpotong
3. **q/Q:** Dari ACF plot — lihat di lag mana korelasi signifikan lalu terpotong
4. **AIC (Akaike Information Criterion):** Semakin rendah AIC, semakin baik model

Kita akan melakukan **grid search** — mencoba beberapa kombinasi parameter dan memilih yang AIC-nya terendah.

> 💡 **Tips:** Jangan over-fit! Model dengan AIC rendah tapi parameter terlalu banyak bisa generalize buruk.

In [ ]:
def fit_sarima(train_vals, n, period=SEASONAL_PERIOD):
    idx = pd.date_range('2000-01-01', periods=len(train_vals), freq='D')
    series = pd.Series(train_vals, index=idx)

    best_aic   = np.inf
    best_model = None
    best_label = None
    search_log = []

    # Grid search
    orders = [(1,1,1), (0,1,1), (1,0,1), (1,1,0), (0,1,0)]
    seasonal_orders = [(1,1,1,period), (0,1,1,period), (1,0,1,period)]

    total = len(orders) * len(seasonal_orders)
    tried = 0

    for order in orders:
        for s_order in seasonal_orders:
            tried += 1
            try:
                m = SARIMAX(
                    series,
                    order=order,
                    seasonal_order=s_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(disp=False, maxiter=200)

                label = f"({order[0]},{order[1]},{order[2]})({s_order[0]},{s_order[1]},{s_order[2]},{period})"
                search_log.append({'order': label, 'aic': m.aic, 'status': '✅'})

                if m.aic < best_aic:
                    best_aic   = m.aic
                    best_model = m
                    best_label = label
            except Exception as e:
                label = f"({order[0]},{order[1]},{order[2]})({s_order[0]},{s_order[1]},{s_order[2]},{period})"
                search_log.append({'order': label, 'aic': None, 'status': f'❌ {str(e)[:40]}'})

    # Print search log
    print(f"\nSARIMA Grid Search: {tried} kombinasi dicoba")
    log_df = pd.DataFrame(search_log).sort_values('aic', na_position='last')
    print(log_df.head(10).to_string(index=False))

    if best_model is not None:
        fc = best_model.get_forecast(n)
        pred = np.maximum(fc.predicted_mean.values, 0)
        ci   = fc.conf_int(alpha=0.05)
        print(f"\n🏆 Best SARIMA: {best_label}, AIC={best_aic:.0f}")
        return pred, best_model, ci, best_label
    return None, None, None, None

# Evaluate SARIMA for both targets
sarima_models = {}
for target in TARGETS:
    print(f"\n{'═'*55}")
    print(f"  SARIMA — {target}")
    print(f"{'═'*55}")

    tr = train[target].values.astype(float)
    te = test[target].values.astype(float)

    pred, model, ci, label = fit_sarima(tr, len(te))
    if pred is not None:
        sarima_models[target] = (model, label)
        pred = np.maximum(pred, 0)
        mae  = calc_mae(te, pred)
        rmse = calc_rmse(te, pred)
        wape = calc_wape(te, pred)

        all_results.append({'target': target, 'model': 'SARIMA',
                            'mae': mae, 'rmse': rmse, 'wape': wape})
        print(f"\n{target:15s}  MAE={mae:>10,.2f}  RMSE={rmse:>10,.2f}  WAPE={wape*100:>6.2f}%")

### 📊 SARIMA Diagnostics — Analisis Residual

Setelah model di-fit, kita harus memeriksa **residual** (error prediksi):
1. **Residual plot:** Apakah error acak dan tersebar merata?
2. **Histogram + KDE:** Apakah error berdistribusi normal?
3. **QQ-Plot:** Verifikasi normalitas secara visual
4. **ACF Residual:** Apakah masih ada autokorelasi yang belum tertangkap model?

> 💡 Residual yang baik = **white noise** (acak, mean ≈ 0, tidak ada pola)

In [ ]:
for target in TARGETS:
    if target in sarima_models:
        model, label = sarima_models[target]
        fig = model.plot_diagnostics(figsize=(14, 8))
        fig.suptitle(f'SARIMA Diagnostics: {target} — {label}', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        print()

---
# 🏆 BAGIAN 5 — Model Comparison & Selection

Saatnya membandingkan **semua 6 model** dan memilih yang terbaik per target.

### Logika Pemilihan (Sesuai Master Plan §25):
1. Bandingkan WAPE semua model pada holdout set (28 hari terakhir)
2. **Pilih model dengan WAPE terendah**
3. Jika baseline mengalahkan model kompleks → **tetap pilih baseline**
4. Business value > model complexity

In [ ]:
# ── Full Comparison Table ──
compare_df = pd.DataFrame(all_results)
compare_df['wape_pct'] = compare_df['wape'].apply(lambda x: f"{x*100:.2f}%")

val_period = f"{test['date'].min().date()} to {test['date'].max().date()}"

print("\n" + "═" * 80)
print("  FULL MODEL COMPARISON — HOLDOUT VALIDATION")
print("═" * 80)

for target in TARGETS:
    subset = compare_df[compare_df['target'] == target].sort_values('wape').reset_index(drop=True)
    print(f"\n  Target: {target}")
    print(f"  Validation: {val_period}")
    print(f"  {'Rank':<5s} {'Model':<25s} {'MAE':>12s} {'RMSE':>12s} {'WAPE':>10s} {'Status':>10s}")
    print(f"  {'─'*75}")
    for i, (_, row) in enumerate(subset.iterrows()):
        status = '🏆 BEST' if i == 0 else ''
        print(f"  {i+1:<5d} {row['model']:<25s} {row['mae']:>12,.2f} {row['rmse']:>12,.2f} {row['wape']*100:>8.2f}% {status:>10s}")

In [ ]:
# ── Visual: WAPE Comparison ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, target in enumerate(TARGETS):
    subset = compare_df[compare_df['target'] == target].sort_values('wape', ascending=True)
    colors = ['#4CAF50' if j == 0 else ('#FFC107' if j <= 2 else '#90CAF9')
              for j in range(len(subset))]
    bars = axes[i].barh(range(len(subset)), subset['wape'] * 100, color=colors, edgecolor='white', height=0.6)
    axes[i].set_yticks(range(len(subset)))
    axes[i].set_yticklabels(subset['model'].values)
    axes[i].set_xlabel('WAPE (%)', fontsize=11)
    axes[i].set_title(f'Model Ranking — {target}', fontsize=14, fontweight='bold')
    axes[i].invert_yaxis()

    for j, (_, row) in enumerate(subset.iterrows()):
        axes[i].text(row['wape'] * 100 + 0.5, j, f"{row['wape']*100:.1f}%",
                     va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Select Best Model ──
selected_models = {}
for target in TARGETS:
    subset = compare_df[compare_df['target'] == target].sort_values('wape')
    best = subset.iloc[0]
    selected_models[target] = best['model']
    print(f"🏆 {target}: {best['model']} (WAPE={best['wape']*100:.2f}%, MAE={best['mae']:,.2f})")

print(f"\n💡 Catatan: Jika baseline terpilih, itu artinya data kita memiliki variabilitas tinggi")
print(f"   dan model kompleks tidak memberikan keuntungan signifikan — ini NORMAL untuk dataset")
print(f"   dengan hanya ~5 bulan history dan 8 zero-sales days.")

---
# 🔮 BAGIAN 6 — Final Forecast (28 Hari — Februari 2026)

Sekarang kita **re-fit model terpilih pada SELURUH data (153 hari)** dan forecast 28 hari ke depan.

### Kenapa Re-fit?
Saat evaluasi, kita hanya menggunakan data train (125 hari).
Untuk forecast final, kita gunakan **semua data yang tersedia** (153 hari) agar model
memiliki informasi terbaru dan terlengkap.

### Confidence Interval
Setiap prediksi memiliki **batas bawah** dan **batas atas** (95% CI).
Ini menunjukkan range di mana nilai sebenarnya kemungkinan besar akan jatuh.

In [ ]:
# ── Re-fit & Forecast ──
forecast_dates = pd.date_range('2026-02-01', periods=FORECAST_DAYS, freq='D')
forecast_df = pd.DataFrame({
    'forecast_date': forecast_dates,
    'forecast_horizon_day': range(1, FORECAST_DAYS + 1)
})

for target in TARGETS:
    model_name = selected_models[target]
    vals = df[target].values.astype(float)
    n = FORECAST_DAYS

    print(f"\n{'═'*55}")
    print(f"  Re-fitting {model_name} for {target}")
    print(f"  Training on full dataset: {len(vals)} days")
    print(f"{'═'*55}")

    if model_name == 'Mean Baseline':
        pred = np.full(n, vals.mean())
        std = np.std(vals)
        lower, upper = pred - 1.96*std, pred + 1.96*std
    elif model_name == 'Naive Last-Value':
        pred = np.full(n, vals[-1])
        std = np.std(np.diff(vals))
        lower, upper = pred - 1.96*std, pred + 1.96*std
    elif model_name == 'Seasonal Naive 7D':
        pred = predict_seasonal_naive(vals, n)
        resid = [vals[i] - vals[i-7] for i in range(7, len(vals))]
        std = np.std(resid)
        lower, upper = pred - 1.96*std, pred + 1.96*std
    elif model_name == '7-Day Moving Average':
        pred = np.full(n, vals[-7:].mean())
        std = np.std(vals[-7:])
        lower, upper = pred - 1.96*std, pred + 1.96*std
    elif model_name == 'Holt-Winters':
        pred, hw_model = fit_holt_winters(vals, n)
        std = np.std(hw_model.resid)
        lower, upper = pred - 1.96*std, pred + 1.96*std
    elif model_name == 'SARIMA':
        pred, sarima_model, ci, label = fit_sarima(vals, n)
        if ci is not None:
            lower = ci.iloc[:, 0].values
            upper = ci.iloc[:, 1].values
        else:
            std = np.std(vals)
            lower, upper = pred - 1.96*std, pred + 1.96*std

    # Clip negatives
    pred  = np.maximum(pred, 0)
    lower = np.maximum(lower, 0)
    upper = np.maximum(upper, 0)

    # Assign columns
    if target == 'order_count':
        forecast_df['predicted_order_count'] = np.round(pred).astype(int)
        forecast_df['order_lower_bound']     = np.maximum(np.floor(lower), 0).astype(int)
        forecast_df['order_upper_bound']     = np.ceil(upper).astype(int)
        forecast_df['order_model_name']      = model_name
    else:
        forecast_df['predicted_revenue']     = np.round(pred, 2)
        forecast_df['revenue_lower_bound']   = np.round(np.maximum(lower, 0), 2)
        forecast_df['revenue_upper_bound']   = np.round(upper, 2)
        forecast_df['revenue_model_name']    = model_name

    print(f"  ✅ Forecast generated: {len(pred)} days")
    print(f"  Total: {pred.sum():,.0f} | Daily avg: {pred.mean():,.1f}")

print("\n✅ All forecasts generated!")

### 📊 Visualisasi: Actual + Forecast

Chart paling penting untuk dashboard dan portofolio: gabungan data historis (biru) dengan forecast (oranye) dan confidence band (abu-abu).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for i, target in enumerate(TARGETS):
    ax = axes[i]
    model_name = selected_models[target]

    # Historical
    ax.plot(df['date'], df[target], color='#2196F3', linewidth=1.2, label='Actual (Historical)', alpha=0.9)

    # Forecast
    if target == 'order_count':
        pred_col = 'predicted_order_count'
        lower_col = 'order_lower_bound'
        upper_col = 'order_upper_bound'
        title = 'Daily Order Count'
        ylabel = 'Orders'
    else:
        pred_col = 'predicted_revenue'
        lower_col = 'revenue_lower_bound'
        upper_col = 'revenue_upper_bound'
        title = 'Daily Revenue (Rp)'
        ylabel = 'Revenue'

    ax.plot(forecast_df['forecast_date'], forecast_df[pred_col],
            color='#FF5722', linewidth=2, label=f'Forecast ({model_name})', marker='o', markersize=3)

    # Confidence interval
    ax.fill_between(forecast_df['forecast_date'],
                    forecast_df[lower_col], forecast_df[upper_col],
                    alpha=0.2, color='#FF5722', label='95% Confidence Interval')

    # Divider line
    ax.axvline(pd.Timestamp('2026-02-01'), color='red', linestyle='--', alpha=0.5, label='Forecast Start')

    ax.set_title(f'{title} — Actual vs Forecast (Feb 2026)', fontsize=14, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12)
    ax.legend(loc='upper left', fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.tight_layout()
plt.show()

### 📊 Forecast Summary — KPI Cards

Ringkasan prediksi untuk Februari 2026:

In [ ]:
print("═" * 55)
print("  📊 FORECAST SUMMARY — February 2026")
print("═" * 55)

total_orders  = forecast_df['predicted_order_count'].sum()
total_revenue = forecast_df['predicted_revenue'].sum()
avg_orders    = forecast_df['predicted_order_count'].mean()
avg_revenue   = forecast_df['predicted_revenue'].mean()

# Bandingkan dengan Januari 2026 (actual)
jan = df[df['date'] >= '2026-01-01']
jan_orders  = jan['order_count'].sum()
jan_revenue = jan['revenue'].sum()

print(f"\n  {'Metric':<30s} {'Jan 2026 (Actual)':>20s} {'Feb 2026 (Forecast)':>22s}")
print(f"  {'─'*75}")
print(f"  {'Total Orders':<30s} {jan_orders:>20,d} {total_orders:>22,d}")
print(f"  {'Total Revenue':<30s} {'Rp'+f'{jan_revenue:,.0f}':>20s} {'Rp'+f'{total_revenue:,.0f}':>22s}")
print(f"  {'Daily Avg Orders':<30s} {jan_orders/31:>20.1f} {avg_orders:>22.1f}")
print(f"  {'Daily Avg Revenue':<30s} {'Rp'+f'{jan_revenue/31:,.0f}':>20s} {'Rp'+f'{avg_revenue:,.0f}':>22s}")

# MoM change
order_change = (total_orders - jan_orders) / jan_orders * 100 if jan_orders > 0 else 0
rev_change   = (total_revenue - jan_revenue) / jan_revenue * 100 if jan_revenue > 0 else 0
print(f"\n  Order MoM change:   {order_change:+.1f}%")
print(f"  Revenue MoM change: {rev_change:+.1f}%")

print(f"\n  Order model:   {forecast_df['order_model_name'].iloc[0]}")
print(f"  Revenue model: {forecast_df['revenue_model_name'].iloc[0]}")

---
# 💾 BAGIAN 7 — Export CSV & Business Interpretation

### Export forecast_daily.csv

File ini akan di-load ke Power BI untuk Page 4 (Forecast & Planning).

In [ ]:
# ── Reorder columns sesuai spec ──
cols = [
    'forecast_date',
    'predicted_order_count', 'predicted_revenue',
    'order_lower_bound',    'order_upper_bound',
    'revenue_lower_bound',  'revenue_upper_bound',
    'order_model_name',     'revenue_model_name',
    'forecast_horizon_day'
]
forecast_export = forecast_df[cols].copy()

# Export
output_path = os.path.join(DATA_DIR, 'forecast_daily.csv')
forecast_export.to_csv(output_path, index=False)
print(f"✅ Exported: {output_path}")
print(f"   Rows: {len(forecast_export)}")
print()
forecast_export.head(10)

### Export forecast_model_evaluation.csv

File ini berisi tabel perbandingan semua model — untuk transparansi dan dokumentasi.

In [ ]:
# ── Build evaluation table ──
eval_rows = []
val_period = f"{test['date'].min().date()} to {test['date'].max().date()}"

for _, row in compare_df.iterrows():
    eval_rows.append({
        'target':           row['target'],
        'model':            row['model'],
        'mae':              round(row['mae'], 2),
        'rmse':             round(row['rmse'], 2),
        'wape_or_smape':    round(row['wape'] * 100, 2),
        'validation_period': val_period,
        'selected_flag':    row['model'] == selected_models[row['target']]
    })

eval_export = pd.DataFrame(eval_rows)

output_path = os.path.join(DATA_DIR, 'forecast_model_evaluation.csv')
eval_export.to_csv(output_path, index=False)
print(f"✅ Exported: {output_path}")
print(f"   Rows: {len(eval_export)}")
print()
eval_export

---
## 📝 Business Interpretation & Caveats

### Interpretasi Bisnis

Berdasarkan forecasting yang telah dilakukan:

1. **Prediksi Februari 2026** memberikan gambaran **workload dan cashflow** yang diharapkan
2. **Confidence interval** menunjukkan range ketidakpastian — semakin lebar band, semakin tidak pasti prediksi
3. Prediksi ini dapat digunakan untuk:
   - **Perencanaan stok** kertas, tinta, dan supplies
   - **Penjadwalan karyawan** berdasarkan prediksi volume order
   - **Proyeksi cashflow** untuk manajemen keuangan

### ⚠️ Caveats (Batasan yang Harus Disampaikan)

1. **Dataset hanya ~5 bulan** — ini sangat pendek untuk time-series forecasting.
   Model belum bisa menangkap pola tahunan (musim ujian, libur nasional, Ramadan, dll.)

2. **8 zero-sales days** — hari tanpa penjualan (kemungkinan libur/tutup) menambah noise.
   Jika pola tutup toko bisa diprediksi, akurasi bisa ditingkatkan.

3. **Confidence interval lebar** — Ini JUJUR menggambarkan ketidakpastian.
   Jangan menyembunyikan confidence interval dari stakeholder.

4. **Forecast bukan garansi** — Ini adalah estimasi terbaik berdasarkan data historis.
   Faktor eksternal (cuaca, kompetitor, event besar) tidak tertangkap oleh model.

### 🚀 Rekomendasi untuk Meningkatkan Akurasi di Masa Depan
- Kumpulkan **minimal 1 tahun data** untuk menangkap seasonality tahunan
- Tambahkan **feature eksternal**: hari libur nasional, cuaca, event sekolah
- Pertimbangkan model **Prophet** atau **LightGBM** jika data sudah cukup panjang
- Lakukan **backtesting berkala** setiap bulan untuk memantau degradasi model

---